In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

print("✓ imports ready")


✓ imports ready


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px

base_url = "https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{}.parquet"

seasons = [2022, 2023, 2024, 2025]
dfs = []

for season in seasons:
    print(f"Loading {season}...")
    url = base_url.format(season)
    df = pd.read_parquet(url)
    dfs.append(df)

pbp = pd.concat(dfs, ignore_index=True)
print(f"✓ Loaded {len(pbp):,} plays across 4 seasons")

Loading 2022...
Loading 2023...
Loading 2024...
Loading 2025...
✓ Loaded 197,362 plays across 4 seasons


In [4]:
# Keep only the columns we need
cols = ['game_id', 'posteam', 'defteam', 'game_seconds_remaining',
        'score_differential', 'down', 'ydstogo', 'yardline_100',
        'posteam_timeouts_remaining', 'home_team', 'result']

nfl_df = pbp[cols].dropna()

# Create target variable — did possession team win?
nfl_df = nfl_df.copy()
nfl_df['win'] = (nfl_df['result'] > 0).astype(int)

print(f"Shape: {nfl_df.shape}")
print(f"Win rate: {nfl_df['win'].mean():.1%}")
print(nfl_df.head())

Shape: (165790, 12)
Win rate: 55.0%
           game_id posteam defteam  game_seconds_remaining  \
2  2022_01_BAL_NYJ     NYJ     BAL                  3596.0   
3  2022_01_BAL_NYJ     NYJ     BAL                  3569.0   
4  2022_01_BAL_NYJ     NYJ     BAL                  3565.0   
5  2022_01_BAL_NYJ     NYJ     BAL                  3541.0   
6  2022_01_BAL_NYJ     NYJ     BAL                  3533.0   

   score_differential  down  ydstogo  yardline_100  \
2                 0.0   1.0     10.0          78.0   
3                 0.0   1.0     10.0          59.0   
4                 0.0   2.0     10.0          59.0   
5                 0.0   3.0      5.0          54.0   
6                 0.0   4.0     15.0          64.0   

   posteam_timeouts_remaining home_team  result  win  
2                         3.0       NYJ     -15    0  
3                         3.0       NYJ     -15    0  
4                         3.0       NYJ     -15    0  
5                         3.0       NYJ     -1

In [5]:
import plotly.express as px

# Win rate by score differential
nfl_df['score_diff_bucket'] = nfl_df['score_differential'].clip(-35, 35)
win_by_score = nfl_df.groupby('score_diff_bucket')['win'].mean().reset_index()

fig = px.line(win_by_score, 
              x='score_diff_bucket', 
              y='win',
              title='NFL win rate by score differential',
              labels={'score_diff_bucket': 'Score differential', 'win': 'Win rate'})
fig.show()

# Win rate by time remaining
nfl_df['time_bucket'] = (nfl_df['game_seconds_remaining'] // 60).astype(int)
win_by_time = nfl_df.groupby('time_bucket')['win'].mean().reset_index()

fig2 = px.line(win_by_time,
               x='time_bucket',
               y='win',
               title='NFL win rate by minutes remaining',
               labels={'time_bucket': 'Minutes remaining', 'win': 'Win rate'})
fig2.show()

In [6]:
# Feature engineering — create the inputs for the model
nfl_df = nfl_df.copy()

# The key interaction — score diff means different things at different times
nfl_df['score_x_time'] = nfl_df['score_differential'] * nfl_df['game_seconds_remaining']

# Is the possession team the home team?
nfl_df['is_home'] = (nfl_df['posteam'] == nfl_df['home_team']).astype(int)

# Final feature list
features = [
    'score_differential',
    'game_seconds_remaining',
    'score_x_time',
    'down',
    'ydstogo',
    'yardline_100',
    'posteam_timeouts_remaining',
    'is_home'
]

X = nfl_df[features]
y = nfl_df['win']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature summary:")
print(X.describe().round(2))

Features shape: (165790, 8)
Target shape: (165790,)

Feature summary:
       score_differential  game_seconds_remaining  score_x_time       down  \
count           165790.00               165790.00     165790.00  165790.00   
mean                -1.36                 1719.01      -1964.80       2.00   
std                 10.13                 1048.11      14035.01       1.01   
min                -56.00                    0.00     -80955.00       1.00   
25%                 -7.00                  804.00      -8841.00       1.00   
50%                  0.00                 1800.00          0.00       2.00   
75%                  4.00                 2606.00       4172.00       3.00   
max                 50.00                 3600.00      75600.00       4.00   

         ydstogo  yardline_100  posteam_timeouts_remaining   is_home  
count  165790.00     165790.00                   165790.00  165790.0  
mean        8.47         50.15                        2.61       0.5  
std         4.

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

Training rows: 132632
Testing rows: 33158


In [8]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
print(f"✓ Model Trained")

✓ Model Trained


In [9]:
from sklearn.metrics import accuracy_score, log_loss
predictions = model.predict(X_test) # Predicts win or loss (0 or 1) for each play
probability = model.predict_proba(X_test) # Predicts actual probabilities (this is your win probability)
print(f"Accuracy: {accuracy_score(y_test, predictions):.1%}")
print(f"Log Loss: {log_loss(y_test, probability):.4f}")

Accuracy: 55.4%
Log Loss: 0.6874


In [10]:

# Filter for the Vikings vs Colts game — biggest comeback in NFL history
game = nfl_df[nfl_df['game_id'] == '2022_15_IND_MIN']

# Sort by time remaining so plays go from start to end of game
game = game.sort_values('game_seconds_remaining', ascending=False)

# Generate win probabilities for every play in the game
# predict_proba returns two columns — probability of loss and probability of win
win_probs = model.predict_proba(game[features])

# Extract just the win probability (second column)
game = game.copy()
game['win_prob'] = win_probs[:, 1]

print(f"Plays in game: {len(game)}")
print(game[['game_seconds_remaining', 'score_differential', 'win_prob']].head(10))

Plays in game: 195
       game_seconds_remaining  score_differential  win_prob
37330                  3592.0                 0.0  0.548492
37331                  3552.0                 0.0  0.548558
37332                  3518.0                 0.0  0.553289
37333                  3479.0                 0.0  0.542756
37334                  3448.0                 0.0  0.548415
37335                  3416.0                 0.0  0.545077
37336                  3375.0                 0.0  0.547105
37337                  3341.0                 0.0  0.542829
37338                  3336.0                 0.0  0.544053
37340                  3299.0                 0.0  0.540189


In [11]:
fig = px.line(game,
              x='game_seconds_remaining',
              y='win_prob',
              title='Vikings vs Colts 2022 — Biggest Comeback in NFL History',
              labels={'game_seconds_remaining': 'Seconds remaining', 'win_prob': 'Win probability'})
fig.update_xaxes(autorange="reversed")
fig.show()


In [12]:
from xgboost import XGBClassifier

In [13]:
xgb_model= XGBClassifier (n_estimators=500, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train,y_train)

print("✓ XGBoost model trained")

✓ XGBoost model trained


In [14]:
xgb_predictions = xgb_model.predict(X_test) # Predicts win or loss (0 or 1) for each play
xgb_probability = xgb_model.predict_proba(X_test) # Predicts actual probabilities (this is your win probability)
print(f"Accuracy: {accuracy_score(y_test, xgb_predictions):.1%}")
print(f"Log Loss: {log_loss(y_test, xgb_probability):.4f}")

Accuracy: 73.8%
Log Loss: 0.4926


In [22]:
# Filter for the Vikings vs Colts game — XGBoost predictions
game_xgb = nfl_df[nfl_df['game_id'] == '2022_15_IND_MIN']

game_xgb = game_xgb.sort_values('game_seconds_remaining', ascending=False)

win_probs = xgb_model.predict_proba(game_xgb[features])

game_xgb = game_xgb.copy()
game_xgb['win_prob'] = win_probs[:, 1]

print(f"Plays in game: {len(game_xgb)}")
print(game_xgb[['game_seconds_remaining', 'score_differential', 'win_prob']].head(10))

Plays in game: 195
       game_seconds_remaining  score_differential  win_prob
37330                  3592.0                 0.0  0.514481
37331                  3552.0                 0.0  0.493026
37332                  3518.0                 0.0  0.530814
37333                  3479.0                 0.0  0.483719
37334                  3448.0                 0.0  0.483371
37335                  3416.0                 0.0  0.452027
37336                  3375.0                 0.0  0.465046
37337                  3341.0                 0.0  0.403344
37338                  3336.0                 0.0  0.409820
37340                  3299.0                 0.0  0.480936


In [24]:
game_xgb['min_win_prob'] = game_xgb.apply(
    lambda row: row['win_prob'] if row['posteam'] == 'MIN' else 1 - row['win_prob'],
    axis=1
)

game_xgb['ind_win_prob'] = 1 - game_xgb['min_win_prob']

print(game_xgb[['game_seconds_remaining', 'score_differential', 'min_win_prob', 'ind_win_prob']].head(10))

       game_seconds_remaining  score_differential  min_win_prob  ind_win_prob
37330                  3592.0                 0.0      0.485519      0.514481
37331                  3552.0                 0.0      0.506974      0.493026
37332                  3518.0                 0.0      0.469186      0.530814
37333                  3479.0                 0.0      0.516281      0.483719
37334                  3448.0                 0.0      0.516629      0.483371
37335                  3416.0                 0.0      0.547973      0.452027
37336                  3375.0                 0.0      0.534954      0.465046
37337                  3341.0                 0.0      0.596656      0.403344
37338                  3336.0                 0.0      0.590180      0.409820
37340                  3299.0                 0.0      0.519064      0.480936


In [25]:
fig = px.line(game_xgb,
              x='game_seconds_remaining',
              y=['min_win_prob', 'ind_win_prob'],
              title='Vikings vs Colts 2022 — Biggest Comeback in NFL History',
              labels={'game_seconds_remaining': 'Seconds remaining', 'value': 'Win probability'})
fig.update_xaxes(autorange="reversed")
fig.show()



In [26]:
# Filter for the Seahawks vs Rams game — XGBoost predictions
game_xgb = nfl_df[nfl_df['game_id'] == '2025_16_LA_SEA']

game_xgb = game_xgb.sort_values('game_seconds_remaining', ascending=False)

win_probs = xgb_model.predict_proba(game_xgb[features])

game_xgb = game_xgb.copy()
game_xgb['win_prob'] = win_probs[:, 1]

print(f"Plays in game: {len(game_xgb)}")
print(game_xgb[['game_seconds_remaining', 'score_differential', 'win_prob']].head(10))

Plays in game: 174
        game_seconds_remaining  score_differential  win_prob
188169                  3591.0                 0.0  0.547535
188170                  3557.0                 0.0  0.531080
188171                  3519.0                 0.0  0.547589
188172                  3516.0                 0.0  0.582401
188173                  3482.0                 0.0  0.511733
188174                  3445.0                 0.0  0.511478
188175                  3413.0                 0.0  0.492451
188176                  3374.0                 0.0  0.478389
188177                  3339.0                 0.0  0.514735
188178                  3295.0                 0.0  0.573542


In [27]:
game_xgb['sea_win_prob'] = game_xgb.apply(
    lambda row: row['win_prob'] if row['posteam'] == 'SEA' else 1 - row['win_prob'],
    axis=1
)

game_xgb['la_win_prob'] = 1 - game_xgb['sea_win_prob']

print(game_xgb[['game_seconds_remaining', 'score_differential', 'sea_win_prob', 'la_win_prob']].head(10))

        game_seconds_remaining  score_differential  sea_win_prob  la_win_prob
188169                  3591.0                 0.0      0.452465     0.547535
188170                  3557.0                 0.0      0.468920     0.531080
188171                  3519.0                 0.0      0.452411     0.547589
188172                  3516.0                 0.0      0.417599     0.582401
188173                  3482.0                 0.0      0.488267     0.511733
188174                  3445.0                 0.0      0.488522     0.511478
188175                  3413.0                 0.0      0.507549     0.492451
188176                  3374.0                 0.0      0.521611     0.478389
188177                  3339.0                 0.0      0.485265     0.514735
188178                  3295.0                 0.0      0.426458     0.573542


In [30]:
fig = px.line(game_xgb,
              x='game_seconds_remaining',
              y=['sea_win_prob', 'la_win_prob'],
              title='Seahawks vs Rams 2025 — Week 16 Comeback (38-37 OT)',
              labels={'game_seconds_remaining': 'Seconds remaining', 'value': 'Win probability'})

fig.update_xaxes(autorange="reversed")

# End of regulation line
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.add_annotation(x=0, y=1.05, text="OT", showarrow=False, 
                   xref="x", yref="paper", font=dict(size=11))

# 30-14 deficit moment
fig.add_vline(x=840, line_dash="dot", line_color="red", opacity=0.5)
fig.add_annotation(x=840, y=1.05, text="30-14 deficit", showarrow=False,
                   xref="x", yref="paper", font=dict(size=11))

# Clean up layout
fig.update_layout(legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01))

fig.show()
